| | |
|---|---|
| **Project** | Advance RAG 30 - RAG Evaluator Dashboard |
| **Topic** | Systematic RAG pipeline evaluation: retrieval, groundedness, cost, latency |
| **Stack** | sentence-transformers (all-MiniLM-L6-v2), NumPy |
| **Focus** | Educational evaluation framework with quantitative metrics |

## Overview

Building a RAG pipeline is only half the job. **Evaluating** it is the other half.

Without systematic evaluation, teams ship RAG systems with hidden failure modes: irrelevant
retrieval, hallucinated answers, excessive latency, or unnecessarily high costs. This notebook
builds a comprehensive RAG evaluation dashboard that measures:

1. **Retrieval quality** -- Are we fetching the right documents? (Precision@K, Recall@K, MRR, NDCG)
2. **Groundedness** -- Is the answer actually supported by the retrieved context?
3. **Answer correctness** -- Does the generated answer match the expected answer?
4. **Latency** -- How fast is each pipeline stage (encoding, retrieval, generation)?
5. **Cost** -- What are the token and compute costs per query?

The result is a reproducible evaluation framework you can apply to any RAG system.

## Learning Goals

1. Understand the key metrics for evaluating RAG pipelines (retrieval, generation, end-to-end)
2. Build a ground-truth benchmark dataset with labeled relevance judgments
3. Implement retrieval quality metrics: Precision@K, Recall@K, MRR, NDCG
4. Implement groundedness scoring using embedding-based claim verification
5. Measure and analyze latency and cost across pipeline stages
6. Identify failure patterns and diagnose root causes

## Problem Statement

Given a RAG pipeline, we need to answer:
- Is retrieval finding the right documents?
- Are generated answers faithful to the retrieved context (not hallucinated)?
- How does quality vary across question types and difficulty levels?
- What are the latency bottlenecks?
- What would this cost at production scale?

We build answers to all of these using a controlled benchmark with known ground truth.

## Why RAG Evaluation Matters

| Without Evaluation | With Evaluation |
|-------------------|----------------|
| "It seems to work" | Precision@3=0.78, Recall@3=0.92 |
| Unknown hallucination rate | Groundedness=0.85, 15% unsupported claims |
| "It's fast enough" | P50=120ms retrieval, P95=340ms generation |
| "Cost should be fine" | $0.012/query at 1000 QPS = $1,037/day |

Systematic evaluation turns vague confidence into actionable numbers.

## Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers', 'numpy'])
print('Dependencies ready.')

## Imports

In [ ]:
import numpy as np
import time
import warnings
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from dataclasses import dataclass, field
import json
warnings.filterwarnings('ignore')
print('Imports complete.')

## Configuration

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
TOP_K = 3
SEED = 42
np.random.seed(SEED)
print(f'Embedding model: {MODEL_NAME}')
print(f'Top-K: {TOP_K}')

## Load Embedding Model

In [ ]:
t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
model_load_time = time.time() - t0
print(f'Model loaded in {model_load_time:.1f}s')
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

## Knowledge Base (Corpus)

We build a 24-document corpus covering 4 domains (AI, Climate, Medicine, Space).
Each document has a unique ID and domain label for evaluation.

In [ ]:
documents = [
    # AI domain (6 docs)
    {'id': 'AI01', 'domain': 'ai', 'text': 'Machine learning models learn patterns from data without being explicitly programmed. Supervised learning uses labeled examples, while unsupervised learning discovers hidden structures.'},
    {'id': 'AI02', 'domain': 'ai', 'text': 'Neural networks are composed of layers of interconnected nodes. Deep learning uses many layers to learn hierarchical representations of data.'},
    {'id': 'AI03', 'domain': 'ai', 'text': 'Transformers use self-attention mechanisms to process sequences in parallel. They are the foundation of modern language models like GPT and BERT.'},
    {'id': 'AI04', 'domain': 'ai', 'text': 'Reinforcement learning trains agents to make decisions by rewarding desired behaviors. It has been used to master games like Go and chess.'},
    {'id': 'AI05', 'domain': 'ai', 'text': 'Transfer learning allows models pretrained on large datasets to be fine-tuned for specific tasks with less data. This dramatically reduces training costs.'},
    {'id': 'AI06', 'domain': 'ai', 'text': 'Generative AI creates new content including text, images, code, and music. Large language models like GPT-4 can generate human-quality text across many domains.'},

    # Climate domain (6 docs)
    {'id': 'CL01', 'domain': 'climate', 'text': 'Global temperatures have risen approximately 1.1 degrees Celsius since pre-industrial times. The Paris Agreement aims to limit warming to 1.5 degrees.'},
    {'id': 'CL02', 'domain': 'climate', 'text': 'Carbon dioxide concentrations in the atmosphere have exceeded 420 parts per million, the highest level in 800,000 years. Fossil fuel combustion is the primary source.'},
    {'id': 'CL03', 'domain': 'climate', 'text': 'Renewable energy sources including solar and wind now provide over 30% of global electricity. Battery storage technology is solving the intermittency challenge.'},
    {'id': 'CL04', 'domain': 'climate', 'text': 'Arctic sea ice has declined by approximately 13% per decade since 1979. This accelerates warming through reduced albedo effect.'},
    {'id': 'CL05', 'domain': 'climate', 'text': 'Carbon capture and storage technology removes CO2 directly from the atmosphere. Current capacity is about 40 million tonnes per year globally.'},
    {'id': 'CL06', 'domain': 'climate', 'text': 'Sea levels have risen about 20 centimeters since 1900 and the rate is accelerating. Thermal expansion and ice sheet melting are the main drivers.'},

    # Medicine domain (6 docs)
    {'id': 'MD01', 'domain': 'medicine', 'text': 'mRNA vaccines work by instructing cells to produce a protein that triggers an immune response. The COVID-19 vaccines from Pfizer and Moderna used this technology.'},
    {'id': 'MD02', 'domain': 'medicine', 'text': 'CRISPR-Cas9 is a gene editing tool that can precisely modify DNA sequences. It has potential applications in treating genetic diseases like sickle cell anemia.'},
    {'id': 'MD03', 'domain': 'medicine', 'text': 'Antibiotic resistance is a growing global health threat. The WHO estimates that drug-resistant infections could cause 10 million deaths per year by 2050.'},
    {'id': 'MD04', 'domain': 'medicine', 'text': 'AI-assisted diagnostics can detect diseases from medical images with accuracy comparable to specialists. Applications include radiology, pathology, and dermatology.'},
    {'id': 'MD05', 'domain': 'medicine', 'text': 'Telemedicine usage increased by over 3000% during the COVID-19 pandemic. It improves access to healthcare in rural and underserved areas.'},
    {'id': 'MD06', 'domain': 'medicine', 'text': 'Immunotherapy harnesses the body immune system to fight cancer. CAR-T cell therapy has shown remarkable results in treating certain blood cancers.'},

    # Space domain (6 docs)
    {'id': 'SP01', 'domain': 'space', 'text': 'The James Webb Space Telescope has detected the earliest galaxies ever observed, formed just 300 million years after the Big Bang.'},
    {'id': 'SP02', 'domain': 'space', 'text': 'SpaceX Starship is the largest and most powerful rocket ever built. It is designed for missions to Mars and can carry over 100 tonnes to low Earth orbit.'},
    {'id': 'SP03', 'domain': 'space', 'text': 'The Artemis program has returned humans to the Moon for the first time since Apollo. Artemis III included the first woman to walk on the lunar surface.'},
    {'id': 'SP04', 'domain': 'space', 'text': 'Mars rovers Curiosity and Perseverance have found evidence of ancient water on Mars. Perseverance has collected rock samples for future return to Earth.'},
    {'id': 'SP05', 'domain': 'space', 'text': 'Satellite mega-constellations like Starlink provide global internet coverage. Over 5,000 Starlink satellites are currently in orbit.'},
    {'id': 'SP06', 'domain': 'space', 'text': 'The search for exoplanets has identified over 5,500 confirmed planets outside our solar system. Some lie in the habitable zone where liquid water could exist.'},
]

print(f'Corpus: {len(documents)} documents across {len(set(d["domain"] for d in documents))} domains')
for domain in sorted(set(d['domain'] for d in documents)):
    count = sum(1 for d in documents if d['domain'] == domain)
    print(f'  {domain}: {count} docs')

## Embed Corpus

In [ ]:
texts = [d['text'] for d in documents]
t0 = time.time()
doc_embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
encoding_time = time.time() - t0
print(f'Encoded {len(texts)} documents in {encoding_time:.3f}s')
print(f'Avg encoding time per doc: {encoding_time/len(texts)*1000:.1f}ms')

## Benchmark Dataset with Ground Truth

The benchmark contains 12 questions with:
- **Expected relevant document IDs** (for retrieval evaluation)
- **Reference answers** (for answer correctness evaluation)
- **Difficulty levels** (easy/medium/hard)
- **Question types** (factual/comparison/multi-hop)

In [ ]:
benchmark = [
    # Easy factual questions (single-doc, direct match)
    {'qid': 'Q01', 'query': 'How do mRNA vaccines work?',
     'relevant_ids': ['MD01'], 'difficulty': 'easy', 'type': 'factual',
     'reference_answer': 'mRNA vaccines instruct cells to produce a protein that triggers an immune response. The Pfizer and Moderna COVID-19 vaccines used this technology.'},

    {'qid': 'Q02', 'query': 'What is the James Webb Space Telescope used for?',
     'relevant_ids': ['SP01'], 'difficulty': 'easy', 'type': 'factual',
     'reference_answer': 'The James Webb Space Telescope detects the earliest galaxies ever observed, formed just 300 million years after the Big Bang.'},

    {'qid': 'Q03', 'query': 'What are transformers in deep learning?',
     'relevant_ids': ['AI03'], 'difficulty': 'easy', 'type': 'factual',
     'reference_answer': 'Transformers use self-attention mechanisms to process sequences in parallel. They are the foundation of models like GPT and BERT.'},

    {'qid': 'Q04', 'query': 'How much have sea levels risen?',
     'relevant_ids': ['CL06'], 'difficulty': 'easy', 'type': 'factual',
     'reference_answer': 'Sea levels have risen about 20 centimeters since 1900 due to thermal expansion and ice sheet melting.'},

    # Medium questions (may need context from related docs)
    {'qid': 'Q05', 'query': 'How is AI used in medical diagnosis?',
     'relevant_ids': ['MD04', 'AI02'], 'difficulty': 'medium', 'type': 'factual',
     'reference_answer': 'AI-assisted diagnostics detect diseases from medical images with specialist-level accuracy, used in radiology, pathology, and dermatology. This relies on deep neural networks.'},

    {'qid': 'Q06', 'query': 'What renewable energy solutions address climate change?',
     'relevant_ids': ['CL03', 'CL05'], 'difficulty': 'medium', 'type': 'factual',
     'reference_answer': 'Solar and wind energy provide over 30% of global electricity with battery storage solving intermittency. Carbon capture technology removes about 40 million tonnes CO2 per year.'},

    {'qid': 'Q07', 'query': 'What missions have explored Mars?',
     'relevant_ids': ['SP04', 'SP02'], 'difficulty': 'medium', 'type': 'factual',
     'reference_answer': 'Curiosity and Perseverance rovers found ancient water evidence and collected rock samples. SpaceX Starship is designed for future crewed Mars missions.'},

    {'qid': 'Q08', 'query': 'How does gene editing treat diseases?',
     'relevant_ids': ['MD02'], 'difficulty': 'medium', 'type': 'factual',
     'reference_answer': 'CRISPR-Cas9 precisely modifies DNA sequences and has potential to treat genetic diseases like sickle cell anemia.'},

    # Hard questions (multi-hop, cross-domain, or comparison)
    {'qid': 'Q09', 'query': 'Compare supervised and reinforcement learning approaches',
     'relevant_ids': ['AI01', 'AI04'], 'difficulty': 'hard', 'type': 'comparison',
     'reference_answer': 'Supervised learning uses labeled examples to learn patterns, while reinforcement learning trains agents through reward signals. RL has mastered games like Go and chess.'},

    {'qid': 'Q10', 'query': 'How do technology advances affect both healthcare access and space exploration?',
     'relevant_ids': ['MD05', 'SP05'], 'difficulty': 'hard', 'type': 'multi-hop',
     'reference_answer': 'Telemedicine improved healthcare access in rural areas with 3000% usage increase during COVID. Satellite mega-constellations like Starlink provide global internet enabling remote connectivity.'},

    {'qid': 'Q11', 'query': 'What evidence exists for environmental changes and their measurement?',
     'relevant_ids': ['CL01', 'CL04', 'CL06'], 'difficulty': 'hard', 'type': 'multi-hop',
     'reference_answer': 'Global temperatures rose 1.1C since pre-industrial times. Arctic sea ice declined 13% per decade since 1979. Sea levels rose 20cm since 1900 due to thermal expansion and ice melt.'},

    {'qid': 'Q12', 'query': 'How are generative AI and transfer learning related?',
     'relevant_ids': ['AI06', 'AI05'], 'difficulty': 'hard', 'type': 'comparison',
     'reference_answer': 'Generative AI creates new content using large language models. Transfer learning enables these models by pretraining on large datasets then fine-tuning for specific tasks with less data.'},
]

print(f'Benchmark: {len(benchmark)} questions')
for diff in ['easy', 'medium', 'hard']:
    count = sum(1 for q in benchmark if q['difficulty'] == diff)
    print(f'  {diff}: {count} questions')
for qtype in ['factual', 'comparison', 'multi-hop']:
    count = sum(1 for q in benchmark if q['type'] == qtype)
    print(f'  {qtype}: {count} questions')

## Retrieval Function with Timing

Every call records latency for performance analysis.

In [ ]:
def retrieve(query, top_k=TOP_K):
    """Retrieve top-k documents with timing."""
    # Encoding latency
    t0 = time.time()
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    encode_time = time.time() - t0
    
    # Search latency
    t1 = time.time()
    sims = np.dot(doc_embeddings, q_emb)
    top_indices = np.argsort(sims)[::-1][:top_k]
    search_time = time.time() - t1
    
    results = []
    for idx in top_indices:
        results.append({
            'id': documents[idx]['id'],
            'domain': documents[idx]['domain'],
            'text': documents[idx]['text'],
            'score': float(sims[idx]),
            'rank': len(results) + 1
        })
    
    return results, {'encode_ms': encode_time * 1000, 'search_ms': search_time * 1000}

# Quick test
test_r, test_t = retrieve('How do vaccines work?')
print(f'Test retrieval: top-1 = {test_r[0]["id"]} (score={test_r[0]["score"]:.4f})')
print(f'Timing: encode={test_t["encode_ms"]:.1f}ms, search={test_t["search_ms"]:.3f}ms')

## Retrieval Quality Metrics

We implement the standard IR evaluation metrics:

| Metric | Formula | Intuition |
|--------|---------|----------|
| **Precision@K** | relevant_in_top_k / K | What fraction of retrieved docs are relevant? |
| **Recall@K** | relevant_in_top_k / total_relevant | What fraction of all relevant docs did we find? |
| **MRR** | 1 / rank_of_first_relevant | How early does the first relevant doc appear? |
| **NDCG@K** | DCG@K / ideal_DCG@K | Quality-weighted ranking with position discount |

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of top-k results that are relevant."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for rid in top_k if rid in relevant_ids)
    return hits / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of relevant docs found in top-k."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for rid in top_k if rid in relevant_ids)
    return hits / len(relevant_ids) if relevant_ids else 0.0

def mrr(retrieved_ids, relevant_ids):
    """Reciprocal rank of the first relevant result."""
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved_ids, relevant_ids, k):
    """Normalized Discounted Cumulative Gain."""
    def dcg(rels, k):
        return sum(r / np.log2(i + 2) for i, r in enumerate(rels[:k]))
    
    # Actual gains
    gains = [1.0 if rid in relevant_ids else 0.0 for rid in retrieved_ids[:k]]
    actual_dcg = dcg(gains, k)
    
    # Ideal gains
    ideal_gains = sorted(gains, reverse=True)
    ideal_dcg = dcg(ideal_gains, k)
    
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

# Verify with a known example
test_retrieved = ['AI03', 'AI01', 'CL02']
test_relevant = ['AI03']
print(f'Precision@3: {precision_at_k(test_retrieved, test_relevant, 3):.3f} (expected 0.333)')
print(f'Recall@3:    {recall_at_k(test_retrieved, test_relevant, 3):.3f} (expected 1.000)')
print(f'MRR:         {mrr(test_retrieved, test_relevant):.3f} (expected 1.000)')
print(f'NDCG@3:      {ndcg_at_k(test_retrieved, test_relevant, 3):.3f}')

## Groundedness Scoring

**Groundedness** measures whether the generated answer is actually supported by the
retrieved context, or if the system hallucinated content.

We use an embedding-based approach:
1. Split the answer into claims (sentences)
2. For each claim, compute max similarity to any retrieved document
3. A claim is "grounded" if similarity exceeds a threshold
4. Groundedness = fraction of grounded claims

In [ ]:
GROUNDEDNESS_THRESHOLD = 0.55

def score_groundedness(answer, context_texts):
    """Score how well the answer is grounded in the retrieved context."""
    # Split answer into claims (sentences)
    claims = [s.strip() for s in answer.replace('!', '.').replace('?', '.').split('.') if len(s.strip()) > 10]
    if not claims:
        return {'groundedness': 0.0, 'claims': [], 'num_claims': 0}
    
    # Encode claims and context
    claim_embs = model.encode(claims, normalize_embeddings=True, show_progress_bar=False)
    context_embs = model.encode(context_texts, normalize_embeddings=True, show_progress_bar=False)
    
    # For each claim, find max similarity to any context document
    claim_results = []
    for i, claim in enumerate(claims):
        sims = np.dot(context_embs, claim_embs[i])
        max_sim = float(np.max(sims))
        best_ctx_idx = int(np.argmax(sims))
        grounded = max_sim >= GROUNDEDNESS_THRESHOLD
        claim_results.append({
            'claim': claim,
            'max_similarity': max_sim,
            'grounded': grounded,
            'best_context_idx': best_ctx_idx
        })
    
    grounded_count = sum(1 for c in claim_results if c['grounded'])
    groundedness = grounded_count / len(claims)
    
    return {
        'groundedness': groundedness,
        'claims': claim_results,
        'num_claims': len(claims),
        'grounded_claims': grounded_count
    }

# Test groundedness
test_answer = 'mRNA vaccines instruct cells to produce proteins. They were used for COVID-19 by Pfizer.'
test_ctx = ['mRNA vaccines work by instructing cells to produce a protein that triggers an immune response. The COVID-19 vaccines from Pfizer and Moderna used this technology.']
g_result = score_groundedness(test_answer, test_ctx)
print(f'Test groundedness: {g_result["groundedness"]:.3f} ({g_result["grounded_claims"]}/{g_result["num_claims"]} claims grounded)')

## Answer Correctness Scoring

Answer correctness measures semantic similarity between the generated answer
and the reference (expected) answer. We use embedding cosine similarity.

In [ ]:
def score_answer_correctness(generated_answer, reference_answer):
    """Semantic similarity between generated and reference answer."""
    embs = model.encode([generated_answer, reference_answer], normalize_embeddings=True, show_progress_bar=False)
    sim = float(np.dot(embs[0], embs[1]))
    return sim

# Test
test_sim = score_answer_correctness(
    'mRNA vaccines make cells produce proteins to trigger immunity.',
    'mRNA vaccines instruct cells to produce a protein that triggers an immune response.'
)
print(f'Test answer correctness: {test_sim:.4f}')

## Answer Generation

Since we do not use an external LLM, we build a rule-based answer synthesizer
that extracts and combines key sentences from retrieved documents.
In production, this would be handled by an LLM like Qwen3-Instruct.

This is sufficient for demonstrating the evaluation framework.

In [ ]:
def generate_answer(query, retrieved_docs):
    """Generate answer by extracting relevant sentences from retrieved docs."""
    t0 = time.time()
    
    # Simple extractive approach: use the top documents' text
    q_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    
    # Score individual sentences
    all_sentences = []
    for doc in retrieved_docs:
        for sent in doc['text'].split('. '):
            sent = sent.strip().rstrip('.')
            if len(sent) > 15:
                all_sentences.append(sent)
    
    if not all_sentences:
        return 'No relevant information found.', time.time() - t0
    
    sent_embs = model.encode(all_sentences, normalize_embeddings=True, show_progress_bar=False)
    sims = np.dot(sent_embs, q_emb)
    
    # Pick top 2-3 most relevant sentences
    top_sent_indices = np.argsort(sims)[::-1][:min(3, len(all_sentences))]
    answer_parts = [all_sentences[i] for i in sorted(top_sent_indices)]
    answer = '. '.join(answer_parts) + '.'
    
    gen_time = time.time() - t0
    return answer, gen_time

# Test
test_docs, _ = retrieve('How do mRNA vaccines work?')
test_ans, test_gen_t = generate_answer('How do mRNA vaccines work?', test_docs)
print(f'Generated answer: {test_ans[:120]}...')
print(f'Generation time: {test_gen_t*1000:.1f}ms')

## Run Full Pipeline Evaluation

In [ ]:
evaluation_results = []

print('Running evaluation on benchmark...')
print('=' * 90)

for q in benchmark:
    # Retrieve
    results, timings = retrieve(q['query'], top_k=TOP_K)
    retrieved_ids = [r['id'] for r in results]
    context_texts = [r['text'] for r in results]
    
    # Generate answer
    answer, gen_time = generate_answer(q['query'], results)
    
    # Retrieval metrics
    p_at_k = precision_at_k(retrieved_ids, q['relevant_ids'], TOP_K)
    r_at_k = recall_at_k(retrieved_ids, q['relevant_ids'], TOP_K)
    mrr_score = mrr(retrieved_ids, q['relevant_ids'])
    ndcg_score = ndcg_at_k(retrieved_ids, q['relevant_ids'], TOP_K)
    
    # Groundedness
    g_result = score_groundedness(answer, context_texts)
    
    # Answer correctness
    correctness = score_answer_correctness(answer, q['reference_answer'])
    
    # Total latency
    total_latency_ms = timings['encode_ms'] + timings['search_ms'] + gen_time * 1000
    
    row = {
        'qid': q['qid'],
        'difficulty': q['difficulty'],
        'type': q['type'],
        'precision_at_k': p_at_k,
        'recall_at_k': r_at_k,
        'mrr': mrr_score,
        'ndcg_at_k': ndcg_score,
        'groundedness': g_result['groundedness'],
        'grounded_claims': g_result['grounded_claims'],
        'total_claims': g_result['num_claims'],
        'correctness': correctness,
        'encode_ms': timings['encode_ms'],
        'search_ms': timings['search_ms'],
        'gen_ms': gen_time * 1000,
        'total_ms': total_latency_ms,
        'retrieved_ids': retrieved_ids,
        'relevant_ids': q['relevant_ids'],
        'answer': answer
    }
    evaluation_results.append(row)
    
    # Print per-query summary
    hit = 'HIT' if mrr_score > 0 else 'MISS'
    print(f'{q["qid"]} [{hit}] P@{TOP_K}={p_at_k:.2f} R@{TOP_K}={r_at_k:.2f} MRR={mrr_score:.2f} '
          f'Ground={g_result["groundedness"]:.2f} Correct={correctness:.2f} Latency={total_latency_ms:.0f}ms')

print('\n' + '=' * 90)
print('Evaluation complete.')

## Aggregate Metrics Dashboard

In [ ]:
n = len(evaluation_results)

# Retrieval metrics
avg_precision = float(np.mean([r['precision_at_k'] for r in evaluation_results]))
avg_recall = float(np.mean([r['recall_at_k'] for r in evaluation_results]))
avg_mrr = float(np.mean([r['mrr'] for r in evaluation_results]))
avg_ndcg = float(np.mean([r['ndcg_at_k'] for r in evaluation_results]))

# Generation metrics
avg_groundedness = float(np.mean([r['groundedness'] for r in evaluation_results]))
avg_correctness = float(np.mean([r['correctness'] for r in evaluation_results]))

# Latency
latencies = [r['total_ms'] for r in evaluation_results]
p50_latency = float(np.percentile(latencies, 50))
p95_latency = float(np.percentile(latencies, 95))
avg_latency = float(np.mean(latencies))

print('RAG EVALUATION DASHBOARD')
print('=' * 55)
print()
print('RETRIEVAL QUALITY')
print(f'  Precision@{TOP_K}:     {avg_precision:.3f}')
print(f'  Recall@{TOP_K}:        {avg_recall:.3f}')
print(f'  MRR:              {avg_mrr:.3f}')
print(f'  NDCG@{TOP_K}:          {avg_ndcg:.3f}')
print()
print('GENERATION QUALITY')
print(f'  Groundedness:     {avg_groundedness:.3f}')
print(f'  Correctness:      {avg_correctness:.3f}')
print()
print('LATENCY (ms)')
print(f'  P50:              {p50_latency:.1f}')
print(f'  P95:              {p95_latency:.1f}')
print(f'  Mean:             {avg_latency:.1f}')
print()
print('COST ESTIMATION (per 1000 queries)')
avg_tokens_per_query = 150  # approximate: query + context + answer
cost_per_1k_tokens = 0.002  # typical for small models
total_tokens_1k = avg_tokens_per_query * 1000
estimated_cost_1k = total_tokens_1k / 1000 * cost_per_1k_tokens
compute_seconds_1k = avg_latency / 1000 * 1000
print(f'  Avg tokens/query: ~{avg_tokens_per_query}')
print(f'  Token cost (1K):  ~${estimated_cost_1k:.2f}')
print(f'  Compute time:     ~{compute_seconds_1k:.1f}s')

## Breakdown by Difficulty Level

In [ ]:
print(f'{"Difficulty":>10s} | {"P@K":>6s} | {"R@K":>6s} | {"MRR":>6s} | {"Ground":>7s} | {"Correct":>8s} | {"Lat(ms)":>8s} | Count')
print('-' * 80)

for diff in ['easy', 'medium', 'hard']:
    subset = [r for r in evaluation_results if r['difficulty'] == diff]
    if not subset:
        continue
    p = float(np.mean([r['precision_at_k'] for r in subset]))
    r = float(np.mean([r['recall_at_k'] for r in subset]))
    m = float(np.mean([r['mrr'] for r in subset]))
    g = float(np.mean([r['groundedness'] for r in subset]))
    c = float(np.mean([r['correctness'] for r in subset]))
    l = float(np.mean([r['total_ms'] for r in subset]))
    print(f'{diff:>10s} | {p:>6.3f} | {r:>6.3f} | {m:>6.3f} | {g:>7.3f} | {c:>8.3f} | {l:>8.1f} | {len(subset)}')

## Breakdown by Question Type

In [ ]:
print(f'{"Type":>12s} | {"P@K":>6s} | {"R@K":>6s} | {"MRR":>6s} | {"Ground":>7s} | {"Correct":>8s} | Count')
print('-' * 70)

for qtype in ['factual', 'comparison', 'multi-hop']:
    subset = [r for r in evaluation_results if r['type'] == qtype]
    if not subset:
        continue
    p = float(np.mean([r['precision_at_k'] for r in subset]))
    r = float(np.mean([r['recall_at_k'] for r in subset]))
    m = float(np.mean([r['mrr'] for r in subset]))
    g = float(np.mean([r['groundedness'] for r in subset]))
    c = float(np.mean([r['correctness'] for r in subset]))
    print(f'{qtype:>12s} | {p:>6.3f} | {r:>6.3f} | {m:>6.3f} | {g:>7.3f} | {c:>8.3f} | {len(subset)}')

## Per-Query Detailed Results

In [ ]:
print(f'{"QID":>4s} | {"Diff":>6s} | {"Type":>10s} | {"P@K":>5s} | {"R@K":>5s} | {"MRR":>5s} | {"Grnd":>5s} | {"Corr":>5s} | {"ms":>6s} | Retrieved')
print('-' * 100)
for r in evaluation_results:
    print(f'{r["qid"]:>4s} | {r["difficulty"]:>6s} | {r["type"]:>10s} | '
          f'{r["precision_at_k"]:>5.2f} | {r["recall_at_k"]:>5.2f} | {r["mrr"]:>5.2f} | '
          f'{r["groundedness"]:>5.2f} | {r["correctness"]:>5.2f} | {r["total_ms"]:>6.0f} | '
          f'{r["retrieved_ids"]}')

## Latency Analysis

Understanding where time is spent helps optimize the pipeline.
We break down latency into: query encoding, vector search, and answer generation.

In [ ]:
# Per-stage latency
encode_times = [r['encode_ms'] for r in evaluation_results]
search_times = [r['search_ms'] for r in evaluation_results]
gen_times = [r['gen_ms'] for r in evaluation_results]

print('Latency Breakdown (ms)')
print('=' * 50)
pct_header = '% Total'
print(f'{"Stage":>15s} | {"Mean":>8s} | {"P50":>8s} | {"P95":>8s} | {pct_header:>8s}')
print('-' * 55)

total_mean = float(np.mean(latencies))
for name, times in [('Encoding', encode_times), ('Search', search_times), ('Generation', gen_times)]:
    mean_t = float(np.mean(times))
    p50_t = float(np.percentile(times, 50))
    p95_t = float(np.percentile(times, 95))
    pct = mean_t / total_mean * 100
    print(f'{name:>15s} | {mean_t:>8.1f} | {p50_t:>8.1f} | {p95_t:>8.1f} | {pct:>7.1f}%')

# Identify bottleneck
stages = {'Encoding': np.mean(encode_times), 'Search': np.mean(search_times), 'Generation': np.mean(gen_times)}
bottleneck = max(stages, key=stages.get)
print(f'\nBottleneck: {bottleneck} ({stages[bottleneck]:.1f}ms avg, {stages[bottleneck]/total_mean*100:.0f}% of total)')

## Error Analysis

In [ ]:
print('Error Analysis')
print('=' * 70)

# Retrieval failures (MRR = 0)
retrieval_failures = [r for r in evaluation_results if r['mrr'] == 0]
print(f'\nRetrieval failures (MRR=0): {len(retrieval_failures)}/{n}')
for r in retrieval_failures:
    print(f'  {r["qid"]}: retrieved={r["retrieved_ids"]}, expected={r["relevant_ids"]}')

# Low groundedness
low_ground = [r for r in evaluation_results if r['groundedness'] < 0.7]
print(f'\nLow groundedness (<0.70): {len(low_ground)}/{n}')
for r in low_ground:
    print(f'  {r["qid"]}: groundedness={r["groundedness"]:.2f} ({r["grounded_claims"]}/{r["total_claims"]} claims)')

# Low correctness
low_correct = [r for r in evaluation_results if r['correctness'] < 0.7]
print(f'\nLow correctness (<0.70): {len(low_correct)}/{n}')
for r in low_correct:
    print(f'  {r["qid"]}: correctness={r["correctness"]:.2f}, difficulty={r["difficulty"]}')

# Partial recall (not all relevant docs found)
partial_recall = [r for r in evaluation_results if r['recall_at_k'] < 1.0 and r['recall_at_k'] > 0]
print(f'\nPartial recall (0 < R@K < 1): {len(partial_recall)}/{n}')
for r in partial_recall:
    missed = [rid for rid in r['relevant_ids'] if rid not in r['retrieved_ids']]
    print(f'  {r["qid"]}: R@K={r["recall_at_k"]:.2f}, missed={missed}')

## Groundedness Deep Dive

We examine the claim-level groundedness for the query with the lowest groundedness
score to understand what types of claims fail verification.

In [ ]:
# Find worst groundedness case for deep dive
worst_q = min(evaluation_results, key=lambda r: r['groundedness'])
print(f'Worst groundedness: {worst_q["qid"]} (score={worst_q["groundedness"]:.2f})')
print(f'Query: {[q for q in benchmark if q["qid"]==worst_q["qid"]][0]["query"]}')
print(f'Answer: {worst_q["answer"][:200]}')
print()

# Re-score to get detailed claims
worst_docs, _ = retrieve([q for q in benchmark if q['qid']==worst_q['qid']][0]['query'])
ctx_texts = [d['text'] for d in worst_docs]
detailed = score_groundedness(worst_q['answer'], ctx_texts)

print('Claim-level analysis:')
for i, claim in enumerate(detailed['claims']):
    status = 'GROUNDED' if claim['grounded'] else 'UNGROUNDED'
    print(f'  [{status}] sim={claim["max_similarity"]:.3f} | "{claim["claim"][:80]}"')

# Best groundedness for contrast
best_q = max(evaluation_results, key=lambda r: r['groundedness'])
print(f'\nBest groundedness: {best_q["qid"]} (score={best_q["groundedness"]:.2f})')

## Cost Analysis

Understanding RAG costs helps with capacity planning. We estimate costs
across different scale scenarios.

Cost components:
- **Embedding cost**: encoding queries (local model = compute only)
- **LLM cost**: answer generation (if using API-based LLM)
- **Infrastructure**: vector DB hosting, compute

Since we use a local embedding model, our costs are compute-only.

In [ ]:
# Cost model parameters
EMBEDDING_COST_PER_1K_TOKENS = 0.0001  # local model, ~ electricity cost
LLM_COST_PER_1K_TOKENS = 0.002  # if using GPT-4o-mini equivalent
AVG_QUERY_TOKENS = 20
AVG_CONTEXT_TOKENS = 200  # ~3 docs * ~67 tokens
AVG_ANSWER_TOKENS = 60

print('Cost Analysis by Scale')
print('=' * 75)
print(f'{"Scale":>15s} | {"Queries/day":>12s} | {"Embedding":>10s} | {"LLM":>10s} | {"Total/day":>10s} | {"Total/mo":>10s}')
print('-' * 75)

for label, qpd in [('Small', 100), ('Medium', 10000), ('Large', 1000000)]:
    embed_tokens = (AVG_QUERY_TOKENS + AVG_CONTEXT_TOKENS) * qpd
    llm_tokens = (AVG_QUERY_TOKENS + AVG_CONTEXT_TOKENS + AVG_ANSWER_TOKENS) * qpd
    embed_cost = embed_tokens / 1000 * EMBEDDING_COST_PER_1K_TOKENS
    llm_cost = llm_tokens / 1000 * LLM_COST_PER_1K_TOKENS
    total_day = embed_cost + llm_cost
    total_mo = total_day * 30
    print(f'{label:>15s} | {qpd:>12,d} | ${embed_cost:>9.2f} | ${llm_cost:>9.2f} | ${total_day:>9.2f} | ${total_mo:>9.0f}')

print()
print('Notes:')
print('  - Embedding costs are minimal with local models (compute/electricity only)')
print('  - LLM costs dominate at scale if using API-based generation')
print('  - Self-hosted LLMs eliminate per-token costs but add infrastructure costs')
print(f'  - At {avg_latency:.0f}ms/query, max throughput ~ {1000/avg_latency:.0f} QPS on single thread')

## Sensitivity Analysis: Effect of Top-K on Retrieval Quality

We test how retrieval metrics change as we vary K from 1 to 6.

In [ ]:
k_values = [1, 2, 3, 4, 5, 6]
k_results = []

for k in k_values:
    k_precisions = []
    k_recalls = []
    k_mrrs = []
    k_ndcgs = []
    
    for q in benchmark:
        results, _ = retrieve(q['query'], top_k=k)
        rids = [r['id'] for r in results]
        k_precisions.append(precision_at_k(rids, q['relevant_ids'], k))
        k_recalls.append(recall_at_k(rids, q['relevant_ids'], k))
        k_mrrs.append(mrr(rids, q['relevant_ids']))
        k_ndcgs.append(ndcg_at_k(rids, q['relevant_ids'], k))
    
    k_results.append({
        'k': k,
        'precision': float(np.mean(k_precisions)),
        'recall': float(np.mean(k_recalls)),
        'mrr': float(np.mean(k_mrrs)),
        'ndcg': float(np.mean(k_ndcgs))
    })

print('Effect of Top-K on retrieval quality:')
print(f'{"K":>3s} | {"P@K":>7s} | {"R@K":>7s} | {"MRR":>7s} | {"NDCG@K":>7s}')
print('-' * 40)
for kr in k_results:
    print(f'{kr["k"]:>3d} | {kr["precision"]:>7.3f} | {kr["recall"]:>7.3f} | {kr["mrr"]:>7.3f} | {kr["ndcg"]:>7.3f}')

# Identify optimal K
# F1-like combination of precision and recall
f1_scores = []
for kr in k_results:
    p, r = kr['precision'], kr['recall']
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    f1_scores.append(f1)
best_k_idx = int(np.argmax(f1_scores))
print(f'\nOptimal K (by P-R F1): K={k_values[best_k_idx]} (F1={f1_scores[best_k_idx]:.3f})')

## Domain-Level Retrieval Analysis

In [ ]:
# Which domains are retrieved most often?
domain_retrieval_counts = defaultdict(int)
domain_correct_counts = defaultdict(int)

for r in evaluation_results:
    q = [q for q in benchmark if q['qid'] == r['qid']][0]
    expected_domains = set()
    for rid in q['relevant_ids']:
        for d in documents:
            if d['id'] == rid:
                expected_domains.add(d['domain'])
    
    for rid in r['retrieved_ids']:
        for d in documents:
            if d['id'] == rid:
                domain_retrieval_counts[d['domain']] += 1
                if d['domain'] in expected_domains:
                    domain_correct_counts[d['domain']] += 1

print('Domain retrieval distribution:')
print(f'{"Domain":>10s} | {"Retrieved":>9s} | {"Correct":>7s} | {"Precision":>9s}')
print('-' * 45)
for domain in sorted(domain_retrieval_counts.keys()):
    total = domain_retrieval_counts[domain]
    correct = domain_correct_counts.get(domain, 0)
    prec = correct / total if total > 0 else 0
    print(f'{domain:>10s} | {total:>9d} | {correct:>7d} | {prec:>9.3f}')

## Understanding the Evaluation Metrics

### Retrieval Metrics

| Metric | What It Measures | Good Range | When Low, Check... |
|--------|-----------------|------------|--------------------|
| **Precision@K** | Are retrieved docs relevant? | > 0.6 | Embedding model quality, chunking strategy |
| **Recall@K** | Did we find all relevant docs? | > 0.8 | K value too small, embedding coverage |
| **MRR** | Is the best doc ranked first? | > 0.8 | Similarity scoring, re-ranking needed |
| **NDCG@K** | Overall ranking quality | > 0.7 | Combination of precision and ranking |

### Generation Metrics

| Metric | What It Measures | Good Range | When Low, Check... |
|--------|-----------------|------------|--------------------|
| **Groundedness** | Is answer supported by context? | > 0.85 | LLM is hallucinating, context too short |
| **Correctness** | Does answer match reference? | > 0.7 | Retrieval feeding wrong docs, generation quality |

### Latency Metrics

| Metric | What It Measures | Good Range | When High, Check... |
|--------|-----------------|------------|--------------------|
| **P50 latency** | Typical response time | < 500ms | Model size, hardware, batch size |
| **P95 latency** | Worst-case response time | < 2000ms | Cold starts, GC pauses, network |

## Limitations

1. **No LLM for generation** -- extractive answers are less natural than LLM-generated ones; groundedness scores may differ with a real LLM
2. **Embedding-based groundedness** is approximate -- an LLM-as-judge would catch nuanced hallucinations better
3. **Small benchmark** (12 questions) -- production evaluations need 100+ diverse test cases
4. **Synthetic corpus** -- real-world documents are longer, noisier, and more varied
5. **No chunking evaluation** -- we use whole documents; real systems must also evaluate chunk boundaries
6. **Cost estimates are approximate** -- actual costs depend on hosting, caching, and traffic patterns

## Common Mistakes in RAG Evaluation

| Mistake | Why It's Wrong | What To Do Instead |
|---------|---------------|--------------------|
| Evaluating only end-to-end quality | Cannot diagnose whether failure is retrieval or generation | Evaluate retrieval and generation separately |
| Using accuracy on hand-picked examples | Selection bias gives unrealistic scores | Use a diverse benchmark with difficulty levels |
| Ignoring groundedness | Fluent answers can be completely hallucinated | Always check if answer is supported by context |
| Not measuring latency | Users abandon slow responses | Track P50 and P95 latency per stage |
| Using only one retrieval metric | Precision and recall capture different failures | Report Precision, Recall, MRR, and NDCG together |
| Not versioning benchmarks | Cannot compare changes over time | Version-control your benchmark dataset |
| Evaluating in bulk only | Missing per-query failure patterns | Always include per-query error analysis |

## Mini Challenge

1. Add 6 more benchmark questions targeting cross-domain queries and measure how they affect aggregate metrics
2. Implement a re-ranking step (e.g., cross-encoder) between retrieval and generation, then compare metrics before/after
3. Build a regression test: save current metrics as baseline and flag any future run that drops below them
4. Add a "faithfulness" metric that checks whether the answer introduces facts not in the context
5. Implement an LLM-as-judge for groundedness using Qwen3-Instruct and compare with the embedding-based approach

## Production Considerations

| Area | Recommendation |
|------|---------------|
| Benchmark size | 100-500 diverse questions with human relevance judgments |
| Evaluation frequency | Run on every pipeline change; nightly regression suite |
| Metrics storage | Track metrics over time in a database for trend analysis |
| Alerting | Set thresholds and alert when metrics drop below baseline |
| A/B testing | Compare retrieval strategies on the same benchmark before deploying |
| Human evaluation | Periodically verify automated metrics with human judges |
| Cost monitoring | Track per-query costs and set budget alerts |
| Latency SLOs | Define P50 < 200ms and P95 < 1000ms targets |

## How to Improve This Project

1. **Larger benchmark**: Add 50+ questions across more domains and difficulty levels
2. **LLM-as-judge**: Use Qwen3-Instruct for groundedness and correctness scoring instead of embedding similarity
3. **Statistical significance**: Add bootstrap confidence intervals to all metrics
4. **Chunking evaluation**: Test how different chunk sizes affect retrieval quality
5. **Comparative evaluation**: Evaluate multiple embedding models (MiniLM vs BGE-M3 vs e5) on the same benchmark
6. **Visual dashboard**: Export metrics to a Streamlit dashboard for interactive exploration
7. **Regression tests**: Automatically fail CI/CD if metrics drop below baseline thresholds

## Key Takeaways

1. **Evaluate retrieval and generation separately** -- they have different failure modes and require different fixes
2. **Use multiple retrieval metrics** (Precision, Recall, MRR, NDCG) because each captures a different aspect of quality
3. **Groundedness is critical** -- fluent answers can be completely hallucinated; always verify claims against context
4. **Track latency per stage** to identify bottlenecks: encoding, search, and generation
5. **Cost modeling at scale** prevents surprises: a $0.01/query system costs $10K/day at 1M queries
6. **Difficulty and type breakdowns** reveal where the pipeline struggles and guide targeted improvements
7. **K sensitivity analysis** finds the optimal retrieval depth balancing precision and recall
8. **Error analysis is more valuable than aggregate metrics** -- it tells you exactly what to fix